# COVID-19 theo 3 châu lục: EDA chuỗi thời gian và Machine Learning

## Mục tiêu
- Tổng hợp dữ liệu gốc thành **tổng theo ngày của từng châu lục**: Bắc Mỹ, Châu Á, Châu Phi.
- Khám phá xu hướng, ca mắc mới, ca tích lũy, trung bình trượt 7/14 ngày và so sánh các châu lục.
- Xây dựng và đánh giá 5 mô hình hồi quy dự báo ca mắc mới: Linear Regression, Ridge Regression, Random Forest, Gradient Boosting, Decision Tree.
- Giữ đúng tinh thần file hướng dẫn: feature engineering chuỗi thời gian, chia train/test theo thứ tự thời gian (không shuffle), MAE/RMSE/R²/MAPE, so sánh dự báo và phân tích sai số.

**Lưu ý dữ liệu:** Các file đầu vào có cột `Confirmed`, `Deaths`, `Recovered`, `Active` là số liệu tích lũy theo ngày. Vì vậy, `New_Confirmed` được tính bằng sai phân của tổng `Confirmed` giữa hai ngày liên tiếp. Ngày đầu tiên đặt bằng 0 do không có ngày trước để tính sai phân. Số ca mới âm nếu xuất hiện phản ánh điều chỉnh dữ liệu nguồn; không tự động sửa về 0 khi phân tích.


## PHẦN 1 — Đọc và tổng hợp dữ liệu theo châu lục

Đặt 3 file CSV cùng thư mục với notebook. Nếu dùng Google Colab, hãy tải 3 file lên hoặc sửa đường dẫn trong ô dưới đây.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

FILES = {
    "Bắc Mỹ": "covid19_north_america.csv",
    "Châu Á": "covid19_asia.csv",
    "Châu Phi": "covid19_africa.csv",
}

def aggregate_continent(file_path, continent_name):
    raw = pd.read_csv(file_path, parse_dates=["Date"])
    # Mỗi ngày: cộng số liệu tích lũy của tất cả quốc gia/tỉnh trong file
    agg = (raw.groupby("Date", as_index=False)[
        ["Confirmed", "Deaths", "Recovered", "Active"]
    ].sum().sort_values("Date").reset_index(drop=True))
    # Tính số ca mới từ mức tích lũy; không lấy cumsum lần nữa
    agg["New_Confirmed"] = agg["Confirmed"].diff()
    agg["New_Deaths"] = agg["Deaths"].diff()
    agg["New_Recovered"] = agg["Recovered"].diff()
    agg.loc[0, ["New_Confirmed", "New_Deaths", "New_Recovered"]] = 0
    agg["Rolling_7D"] = agg["New_Confirmed"].rolling(7, min_periods=1).mean()
    agg["Rolling_14D"] = agg["New_Confirmed"].rolling(14, min_periods=1).mean()
    agg["Continent"] = continent_name
    return agg

frames = []
for continent, file_path in FILES.items():
    part = aggregate_continent(file_path, continent)
    frames.append(part)
    print(continent, "|", part["Date"].min().date(), "đến", part["Date"].max().date(),
          "| số ngày:", len(part), "| tổng Confirmed cuối kỳ:", f'{part["Confirmed"].iloc[-1]:,}')

df = pd.concat(frames, ignore_index=True)
df.to_csv("covid19_continent_daily_totals.csv", index=False)
df.head()


### Giải thích cách tổng hợp
- `Confirmed`, `Deaths`, `Recovered`, `Active`: cộng theo `Date` để có tổng số liệu tích lũy của cả châu lục.
- `New_Confirmed`: lấy `Confirmed hôm nay - Confirmed hôm qua`. Đây mới là biến phù hợp để dự báo số ca mắc mới mỗi ngày.
- Không cộng trực tiếp `Confirmed` qua tất cả ngày, vì như vậy sẽ đếm lặp số ca tích lũy.


## PHẦN 2 — EDA: khám phá dữ liệu

Các biểu đồ dưới đây làm theo cấu trúc hướng dẫn: xu hướng theo thời gian, trung bình trượt, đường tích lũy, so sánh theo châu lục và tương quan.


In [ ]:
# 2.1. Ca mắc mới mỗi ngày và trung bình trượt 7 ngày
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
for ax, continent in zip(axes, FILES.keys()):
    part = df[df["Continent"] == continent]
    ax.bar(part["Date"], part["New_Confirmed"], alpha=0.35, label="Ca mắc mới/ngày")
    ax.plot(part["Date"], part["Rolling_7D"], linewidth=2, label="Trung bình trượt 7 ngày")
    ax.plot(part["Date"], part["Rolling_14D"], linewidth=2, linestyle="--", label="Trung bình trượt 14 ngày")
    ax.set_title(continent)
    ax.set_ylabel("Số ca")
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2.2. Tổng ca xác nhận tích lũy theo châu lục
plt.figure(figsize=(13, 6))
for continent in FILES.keys():
    part = df[df["Continent"] == continent]
    plt.plot(part["Date"], part["Confirmed"], linewidth=2, label=continent)
plt.title("COVID-19: Tổng ca xác nhận tích lũy theo châu lục")
plt.xlabel("Ngày"); plt.ylabel("Ca tích lũy"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 2.3. So sánh tổng cuối kỳ và đỉnh ca mắc mới/ngày
summary = (df.sort_values("Date").groupby("Continent").agg(
    Confirmed_cuoi_ky=("Confirmed", "last"),
    Deaths_cuoi_ky=("Deaths", "last"),
    Recovered_cuoi_ky=("Recovered", "last"),
    Dinh_ca_moi=("New_Confirmed", "max"),
    Ngay_dinh_ca_moi=("New_Confirmed", lambda x: df.loc[x.idxmax(), "Date"])
).reset_index())
summary


### Gợi ý viết nhận xét EDA
1. Châu lục nào có tổng `Confirmed` cuối kỳ cao nhất?
2. Châu lục nào có đỉnh `New_Confirmed` cao nhất và đỉnh rơi vào ngày nào?
3. Đường `Rolling_7D` giúp nhìn xu hướng ra sao so với cột ca mắc mới từng ngày?
4. Có giá trị ca mắc mới âm không? Nếu có, nêu đây có thể là điều chỉnh số liệu nguồn.
5. Không so sánh số ca tuyệt đối như rủi ro cá nhân giữa các châu lục nếu không có dữ liệu dân số để chuẩn hóa.


## PHẦN 3 — Feature Engineering cho chuỗi thời gian

Ta dự báo `New_Confirmed` (ca mắc mới/ngày). Các biến trễ chỉ lấy dữ liệu quá khứ. Chia train/test theo thứ tự thời gian, không dùng `shuffle=True`.


In [ ]:
def make_features(one_continent):
    t = one_continent[["Date", "New_Confirmed"]].copy().sort_values("Date").reset_index(drop=True)
    t["Diff_1"] = t["New_Confirmed"].diff()
    for lag in [1, 2, 3, 7]:
        t[f"Lag_{lag}"] = t["New_Confirmed"].shift(lag)
        t[f"Diff_Lag_{lag}"] = t["Diff_1"].shift(lag)
    t["Rolling_Mean_7"] = t["New_Confirmed"].shift(1).rolling(7).mean()
    t["Rolling_Std_7"] = t["New_Confirmed"].shift(1).rolling(7).std()
    t["DayOfWeek"] = t["Date"].dt.dayofweek
    t["IsWeekend"] = (t["DayOfWeek"] >= 5).astype(int)
    t["Trend"] = np.arange(len(t))
    return t.dropna().reset_index(drop=True)

features_level = ["Lag_1", "Lag_2", "Lag_3", "Lag_7", "Rolling_Mean_7", "DayOfWeek", "Trend"]
features_diff = ["Diff_Lag_1", "Diff_Lag_2", "Diff_Lag_3", "Diff_Lag_7", "DayOfWeek", "IsWeekend"]


## PHẦN 4 — Huấn luyện 5 mô hình và đánh giá

Mô hình 1 học trực tiếp giá trị ca mắc mới (Level). Bốn mô hình còn lại học sai phân `Diff_1`, sau đó cộng mức ca ngày trước đó (`Lag_1`) để khôi phục dự báo về số ca. Tất cả đều dùng temporal split 80% đầu làm train, 20% cuối làm test.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) > 1e-8
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

def score_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE_%": safe_mape(y_true, y_pred),
    }

def run_models(continent_df, continent_name):
    ts = make_features(continent_df)
    split = int(len(ts) * 0.80)
    train, test = ts.iloc[:split].copy(), ts.iloc[split:].copy()
    models = {
        "Linear Regression (Level)": (LinearRegression(), features_level, "level"),
        "Ridge Regression (Diff)": (Ridge(alpha=10.0), features_diff, "diff"),
        "Random Forest (Diff)": (RandomForestRegressor(n_estimators=150, max_depth=5, random_state=RANDOM_STATE), features_diff, "diff"),
        "Gradient Boosting (Diff)": (GradientBoostingRegressor(random_state=RANDOM_STATE), features_diff, "diff"),
        "Decision Tree (Diff)": (DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE), features_diff, "diff"),
    }
    rows, predictions = [], {}
    for name, (model, feats, target_type) in models.items():
        target = "New_Confirmed" if target_type == "level" else "Diff_1"
        model.fit(train[feats], train[target])
        raw_pred = model.predict(test[feats])
        pred = raw_pred if target_type == "level" else test["Lag_1"].to_numpy() + raw_pred
        # Dự báo số ca không âm; vẫn giữ số liệu thực tế nguyên gốc để đánh giá minh bạch
        pred = np.maximum(0, pred)
        metrics = score_model(test["New_Confirmed"], pred)
        rows.append({"Continent": continent_name, "Model": name, **metrics,
                     "Train_days": len(train), "Test_days": len(test)})
        predictions[name] = pred
    result = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
    return ts, train, test, result, predictions

all_results = []
model_runs = {}
for continent in FILES.keys():
    cdf = df[df["Continent"] == continent].copy()
    run = run_models(cdf, continent)
    model_runs[continent] = run
    all_results.append(run[3])
results = pd.concat(all_results, ignore_index=True)
results.sort_values(["Continent", "MAE"])


In [ ]:
# So sánh MAE của 5 mô hình trong từng châu lục
plt.figure(figsize=(14, 6))
sns.barplot(data=results, x="Model", y="MAE", hue="Continent")
plt.title("So sánh MAE của 5 mô hình theo châu lục")
plt.xticks(rotation=20, ha="right")
plt.ylabel("MAE (số ca)")
plt.tight_layout()
plt.show()


In [ ]:
# Actual vs predicted trên tập test, tạo một biểu đồ cho mỗi châu lục
for continent, (ts, train, test, result, predictions) in model_runs.items():
    plt.figure(figsize=(14, 6))
    plt.plot(test["Date"], test["New_Confirmed"], color="black", linewidth=2.5, label="Thực tế")
    for model_name, pred in predictions.items():
        plt.plot(test["Date"], pred, linewidth=1.5, label=model_name)
    plt.title(f"Thực tế và dự báo ca mắc mới — {continent}")
    plt.xlabel("Ngày"); plt.ylabel("Ca mắc mới"); plt.legend()
    plt.tight_layout(); plt.show()


## PHẦN 5 — Kết luận và cách diễn giải

- Với từng châu lục, chọn mô hình có **MAE thấp nhất** làm mô hình tốt nhất theo tiêu chí sai số tuyệt đối trung bình; đồng thời xem RMSE, R², MAPE và biểu đồ dự báo.
- MAE/RMSE càng thấp thường càng tốt. R² càng cao thường càng tốt, nhưng có thể âm nếu dự báo kém hơn dự báo bằng trung bình.
- MAPE không ổn định khi giá trị thực bằng 0 hoặc gần 0; trong code, các ngày có giá trị thực bằng 0 được bỏ khỏi phép tính MAPE.
- Không khẳng định mô hình “dự báo chính xác đại dịch” chỉ từ một giai đoạn ngắn. Đây là bài thực hành học máy trên dữ liệu lịch sử, không phải công cụ dự báo y tế chính thức.
- Vì 3 file đều kết thúc ngày 27/07/2020, kết quả chỉ phản ánh giai đoạn dữ liệu có sẵn, không phải toàn bộ đại dịch.
